In [1]:
# 1. Install ultralytics
!pip install ultralytics

from ultralytics import YOLO
import torch
import os
import shutil

# 2. Verify GPU Power
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}\n")

# 3. Securely mount Kaggle dataset using symlinks (prevents Read-Only errors)
source_dir = "/kaggle/input"
root_data_dir = None
for root, dirs, files in os.walk(source_dir):
    if 'train' in dirs:
        root_data_dir = root
        break

if root_data_dir is None:
    raise ValueError("ERROR: Could not find 'train' folder. Did your zip file upload correctly?")

working_dataset = '/kaggle/working/car_parts_dataset'
os.makedirs(working_dataset, exist_ok=True)

for split in ['train', 'valid', 'test']:
    src = os.path.join(root_data_dir, split)
    if os.path.exists(src):
        link_name = 'val' if split == 'valid' else split
        dst = os.path.join(working_dataset, link_name)
        if not os.path.exists(dst):
            os.symlink(src, dst)

print(f"✅ Fast Virtual Dataset Created at: {working_dataset}\n")

# 4. Initialize and Train the YOLO11 Large Model
print(f"{'='*50}")
print("🚀 STARTING TRAINING FOR YOLO11 LARGE...")
print(f"{'='*50}\n")

model = YOLO('yolo11l-cls.pt')

# Train for 30 epochs (proven to get 99.34% accuracy on your data)
model.train(
    data=working_dataset,
    epochs=30,      
    imgsz=224,      
    device=0,       
    project='/kaggle/working',
    name='final_large_model'
)

# 5. Extract the precious weights instantly!
# Once training finishes, this quickly moves the best.pt file directly 
# to the main /kaggle/working/ folder so it's super easy to download.
best_weights_path = '/kaggle/working/final_large_model/weights/best.pt'
easy_download_path = '/kaggle/working/car_parts_large_v1.pt'

if os.path.exists(best_weights_path):
    shutil.copy(best_weights_path, easy_download_path)
    print(f"\n{'='*50}")
    print("🏆 TRAINING COMPLETE! 🏆")
    print("I have copied your best model directly into /kaggle/working/car_parts_large_v1.pt")
    print("Go to the right-hand panel, click Output -> /kaggle/working, and download 'car_parts_large_v1.pt'!")
    print(f"{'='*50}")
else:
    print("\n🚨 ERROR: Training crashed before saving weights. Please check Kaggle logs.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.3 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
GPU Available: True
GPU Name: Tesla T4

✅ Fast Virtual Dataset Created at: /kaggle/working/car_parts_dataset

🚀 STARTING TRAINING FOR YOLO11 LARGE...

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/car_parts_dataset, degrees=0.0, deterministic=True, device=0, dfl=1.5,